# 🎯 [K13_M10] 추천 도우미 완성하기
### Google Colab 실습 환경 (교육훈련·역량개발 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K13/k13_m10_solution.ipynb)


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 추천 도우미 완성하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **전체 기능을 하나로 묶은 교육 콘텐츠 추천 도우미를 완성하고 자체 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **기록 검증, 하이브리드 추천, 신규 전입 대응, 추천 이유, 다양성 조정을 하나의 도우미로 통합하고 추천 카드를 출력 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `추천 도우미 코드, 자체 점검 결과, 사람 결정 안내와 검토용 프롬프트` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 통합·완성 — 도우미를 통합하고 자체 점검을 통과했으며 이수 결정은 사람이 한다는 안내를 명시했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(추천 도우미 코드, 자체 점검 결과, 사람 결정 안내와 검토용 프롬프트)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""수강 기록 검증, 하이브리드 추천, 신규 전입 대응, 추천 이유, 다양성 조정을 하나로 묶은 '교육 콘텐츠 추천 도우미'를 완성하고 자체 점검한다."""
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

# ---- 공통 함수: 점수 계산(인기도·콘텐츠·협업·행렬 분해), 상위 추천, 평가 ----
SIDX = {s: i for i, s in enumerate(SID)}
GLOBAL_MEAN = train["rating"].mean()
_stat = train.groupby("course")["rating"].agg(["mean", "count"]).reindex(CID).fillna({"mean": GLOBAL_MEAN, "count": 0})
POP_SCORE = ((_stat["count"] * _stat["mean"] + 5 * GLOBAL_MEAN) / (_stat["count"] + 5)).values  # 수강 수가 적으면 전체 평균 쪽으로 보정

_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
TFIDF = _vec.fit_transform(courses["title"] + " " + courses["topic"] + " " + courses["desc"])
ITEM_SIM = cosine_similarity(TFIDF)
USER_SIM = cosine_similarity(RV)
np.fill_diagonal(USER_SIM, 0)
_svd = TruncatedSVD(n_components=6, random_state=13)
_U = _svd.fit_transform(RV)
SVD_SCORE = _U @ _svd.components_


def pop_scores(sid):
    return POP_SCORE.copy()


def content_scores(sid):
    """평점 4 이상으로 수강한 콘텐츠의 설명과 비슷한 콘텐츠일수록 높은 점수"""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (ITEM_SIM * w).sum(axis=1) / w.sum()


def cf_scores(sid, n=10):
    """나와 평점 패턴이 비슷한 장병 n명이 높게 준 콘텐츠일수록 높은 점수"""
    i = SIDX[sid]
    nb = np.argsort(-USER_SIM[i])[:n]
    return (USER_SIM[i, nb][:, None] * RV[nb]).sum(axis=0) / USER_SIM[i, nb].sum()


def svd_scores(sid):
    return SVD_SCORE[SIDX[sid]].copy()


def norm(v, mask):
    """아직 안 들은 콘텐츠들 사이에서 0~1로 맞춘다"""
    x = v.copy()
    lo, hi = x[mask].min(), x[mask].max()
    return np.where(mask, (x - lo) / (hi - lo + 1e-9), 0)


def hybrid_scores(sid, w=(0.2, 0.4, 0.4)):
    unseen = M[SIDX[sid]] == 0
    return (w[0] * norm(pop_scores(sid), unseen) + w[1] * norm(content_scores(sid), unseen)
            + w[2] * norm(cf_scores(sid), unseen))


def top_k(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)    # 이미 들은 콘텐츠는 제외
    return [CID[j] for j in np.argsort(-s)[:k]]


def evaluate(score_fn, k=5, sids=None):
    """장병마다 상위 k개를 추천해 정밀도(Precision@k), 재현율(Recall@k), 커버리지를 구한다."""
    prec, rec, shown = [], [], set()
    for sid, rel in relevant.items():
        if sids is not None and sid not in sids:
            continue
        rc = top_k(score_fn(sid), sid, k)
        hit = len(set(rc) & rel)
        prec.append(hit / k); rec.append(hit / len(rel)); shown |= set(rc)
    return np.mean(prec), np.mean(rec), len(shown) / len(CID)

# ---- 신규 전입 장병(수강 이력 없음) 대응 ----
MAXLEVEL = {"이병": 1, "일병": 2, "상병": 2, "병장": 3}


def cold_scores(interests, rank):
    """이력이 없을 때는 관심 주제 일치(60%) + 인기도(30%) + 난이도 적합(10%)으로 점수를 매긴다."""
    topic_hit = courses["topic"].isin(interests).values.astype(float)
    level_fit = (courses["level"].values <= MAXLEVEL[rank]).astype(float)
    pop = (POP_SCORE - POP_SCORE.min()) / (POP_SCORE.max() - POP_SCORE.min())
    return 0.6 * topic_hit + 0.3 * pop + 0.1 * level_fit


def top_k_new(scores, k=5, exclude=()):
    s = scores.copy()
    for j, c in enumerate(CID):
        if c in exclude:
            s[j] = -1e9
    return [CID[j] for j in np.argsort(-s)[:k]]

# ---- 추천 이유 설명 ----
def explain(sid, cid, n=10):
    """추천 이유를 문장으로 만든다. 다른 장병의 이름은 밝히지 않고 인원수만 말한다."""
    i, j = SIDX[sid], CID.index(cid)
    c, me = courses.iloc[j], soldiers.iloc[i]
    why = []
    if c["topic"] in (me["interest1"], me["interest2"]):
        why.append(f"관심 주제('{c['topic']}')에 해당합니다")
    liked = [k for k in range(len(CID)) if RV[i, k] >= 4]
    if liked:
        k = max(liked, key=lambda x: ITEM_SIM[j, x])
        if ITEM_SIM[j, k] >= 0.15:
            why.append(f"높게 평가한 콘텐츠('{courses.title[k]}')와 설명이 비슷합니다(유사도 {ITEM_SIM[j, k]:.2f})")
    nb = np.argsort(-USER_SIM[i])[:n]
    fans = [x for x in nb if RV[x, j] >= 4]
    if fans:
        why.append(f"수강 패턴이 비슷한 장병 {n}명 중 {len(fans)}명이 높게 평가했습니다")
    if c["level"] <= MAXLEVEL[me["rank"]]:
        why.append(f"{me['rank']} 수준에 맞는 {LEVELS[c['level']]} 과정입니다")
    if not why:
        why.append("전체 장병에게 인기가 높은 과정입니다")
    return why

# ---- 다양성: 같은 주제가 몰리지 않게 다시 정렬 ----
def diversify(scores, sid, k=5, max_per_topic=2):
    """점수 순으로 고르되, 같은 주제는 최대 max_per_topic개까지만 담는다."""
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)
    out, cnt = [], {}
    for j in np.argsort(-s):
        t = courses.topic[j]
        if cnt.get(t, 0) >= max_per_topic:
            continue
        out.append(CID[j]); cnt[t] = cnt.get(t, 0) + 1
        if len(out) == k:
            break
    return out


def topic_count(rc):
    return len({courses.topic[CID.index(c)] for c in rc})


def list_diversity(rc):
    """추천 목록 안 콘텐츠끼리의 설명이 얼마나 서로 다른지(1 - 평균 유사도)"""
    idx = [CID.index(c) for c in rc]
    sub = ITEM_SIM[np.ix_(idx, idx)]
    return 1 - (sub.sum() - len(idx)) / (len(idx) * (len(idx) - 1))

NOTICE = "※ 이 추천은 참고용 제안입니다. 교육 이수 여부와 일정은 본인의 의사와 지휘관·교육 담당자의 확인으로 결정합니다."


class EduRecommender:
    def __init__(self, w=(0.1, 0.6, 0.3)):
        self.w = w

    def validate(self, rows):
        """새 수강 기록의 오류를 걸러 낸다."""
        ok, bad = [], []
        for sid, cid, r in rows:
            if sid in SIDX and cid in CID and 1 <= r <= 5:
                ok.append((sid, cid, r))
            else:
                bad.append((sid, cid, r))
        return ok, bad

    def recommend(self, sid=None, profile=None, k=5):
        """기존 장병(sid)은 하이브리드, 신규 전입(profile: 관심·계급)은 관심 주제 기반으로 추천한다."""
        if sid is not None and sid in SIDX and M[SIDX[sid]].sum() > 0:
            ids = diversify(hybrid_scores(sid, self.w), sid, k, 2)
            return [{"id": c, "title": courses.title[CID.index(c)], "hours": int(courses.hours[CID.index(c)]),
                     "why": explain(sid, c)} for c in ids]
        interests, rank = profile["관심"], profile["계급"]
        ids = top_k_new(cold_scores(interests, rank), k)
        out = []
        for c in ids:
            j = CID.index(c)
            why = [f"관심 주제('{courses.topic[j]}')에 해당합니다"] if courses.topic[j] in interests else ["전체 장병에게 인기가 높은 과정입니다"]
            why.append("수강 이력이 없어 관심 주제와 인기도로 추천했습니다")
            out.append({"id": c, "title": courses.title[j], "hours": int(courses.hours[j]), "why": why})
        return out

    def card(self, items):
        lines = [f"  {n}. {x['title']} ({x['hours']}시간) - {x['why'][0]}" for n, x in enumerate(items, 1)]
        return "\n".join(lines) + "\n  " + NOTICE


R_ = EduRecommender()
print("[기존 장병 S31 추천]")
print(R_.card(R_.recommend("S31")))
print("\n[신규 전입 장병(이력 없음) 추천]")
newp = {"관심": ["체력", "리더십"], "계급": "이병"}
print(R_.card(R_.recommend(profile=newp)))

t0 = time.time()
all_recs = {s: R_.recommend(s) for s in SID}
elapsed = time.time() - t0
ok_rows, bad_rows = R_.validate([("S01", "C01", 4), ("S99", "C01", 3), ("S02", "C99", 3), ("S03", "C02", 9)])
hit = sum(len({x["id"] for x in all_recs[s]} & relevant[s]) for s in relevant)
prec = hit / (5 * len(relevant))
pop_prec = evaluate(pop_scores)[0]
new_recs = R_.recommend(profile=newp)

checks = {
    "잘못된 수강 기록 3건 차단": len(ok_rows) == 1 and len(bad_rows) == 3,
    "이미 수강한 콘텐츠를 추천하지 않음": all(M[SIDX[s], CID.index(x["id"])] == 0 for s in SID for x in all_recs[s]),
    "장병 60명 모두 5개 추천": all(len(v) == 5 for v in all_recs.values()),
    "모든 추천에 이유 제시": all(x["why"] for v in all_recs.values() for x in v),
    "추천마다 서로 다른 주제 3개 이상": all(topic_count([x["id"] for x in v]) >= 3 for v in all_recs.values()),
    "신규 전입 장병도 5개 추천": len(new_recs) == 5 and all(x["why"] for x in new_recs),
    "Precision@5가 인기도의 2배 이상": prec >= 2 * pop_prec,
    "전체 처리 5초 이내": elapsed < 5,
    "다른 장병 ID 미노출": not any(s in " ".join(x["why"]) for v in all_recs.values() for x in v for s in SID),
}
print(f"\n통합 도우미 Precision@5 {prec:.3f} (인기도 {pop_prec:.3f}), 처리 시간 {elapsed:.2f}초")
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {sum(checks.values())}/{len(checks)} 통과")

print("\n[생성형 AI로 추천 목록을 검토할 때 쓰는 프롬프트 예시]")
print("너는 장병 교육 담당자를 돕는 검토자다. 아래 가상 장병의 관심 주제·계급과 추천 목록을 보고, 한 주제에 쏠렸는지, "
      "난이도가 맞는지, 추천 이유가 근거 없이 단정적이지 않은지 지적하라. (가상 자료만 사용)")
print("\n" + NOTICE)


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[K13_M10] 추천 도우미 완성하기 실행 완료
주요 지표 정상 산출 및 PASS 판정 완료.
```
</details>
